# Module 11 — Rule-Based Fraud Heuristics: Velocity & Threshold Detection

> **Track 2 · Revolut Fintech Specialization**  
> **Difficulty**: Intermediate  
> **Runtime**: ~4 minutes  
> **Key Libraries**: Pandas, NumPy, Matplotlib, Seaborn, Plotly  
> **Prerequisite**: Module 1 (Optimized Pandas), Module 2 (GroupBy)

## Table of Contents

1. [Business Context](#1-business-context)
2. [Setup & Imports](#2-setup--imports)
3. [Synthetic Transaction Stream Generation](#3-synthetic-transaction-stream-generation)
4. [Exploratory Data Analysis](#4-exploratory-data-analysis)
5. [Rule 1: Velocity Burst Detection](#5-rule-1-velocity-burst-detection)
6. [Rule 2: Geo-Impossibility Check](#6-rule-2-geo-impossibility-check)
7. [Rule 3: Round-Amount Flag](#7-rule-3-round-amount-flag)
8. [Rule 4: High-Value International](#8-rule-4-high-value-international)
9. [Rule 5: Late-Night ATM Pattern](#9-rule-5-late-night-atm-pattern)
10. [Combined Rule Engine](#10-combined-rule-engine)
11. [Precision/Recall per Rule](#11-precisionrecall-per-rule)
12. [Visualization Dashboard](#12-visualization-dashboard)
13. [Key Business Takeaway](#13-key-business-takeaway)

---
## §1 · Business Context

### Why do rule-based heuristics matter at Revolut?

Before deploying expensive ML models, Revolut's fraud team relies on a **first line of defense**:
a set of **deterministic rules** that flag obviously suspicious transactions in **< 5 ms**.

| Rule Category | Example | Business Impact |
|---------------|---------|-----------------|
| **Velocity** | 5+ ATM withdrawals in 10 minutes | Catches card-skimming rings |
| **Geo-impossibility** | London → New York in 30 minutes | Flags cloned cards |
| **Round amounts** | Exactly £5,000.00 transfer | Indicates structured fraud |
| **High-value international** | £3,000 purchase in Nigeria | High-risk corridor |
| **Late-night ATM** | 3 AM cash withdrawal | Correlates with mugging/drunk spending |

**Why not just use ML?**
- Rules run in **microseconds** (ML models: 10–100 ms).
- Rules are **100% interpretable** for compliance and customer disputes.
- Rules catch **known patterns** that ML may miss in low-data regimes.

**The goal**: build a rule engine that catches **30–40% of fraud** with high precision,
serving as the first filter before ML models score the remaining transactions.

In this notebook we will:
1. Generate a realistic 500K-transaction stream with labeled fraud.
2. Engineer 5 rule-based heuristics used in production.
3. Measure precision/recall for each rule individually and combined.
4. Build an executive-ready rule-performance dashboard.

---
## §2 · Setup & Imports

In [ ]:
# ── Reproducibility ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import time
import warnings
from datetime import timedelta

np.random.seed(42)
warnings.filterwarnings("ignore")

# ── Visualization ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# ── Metrics ──────────────────────────────────────────────────────
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

# ── Display settings ─────────────────────────────────────────────
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
sns.set_theme(style="whitegrid", palette="muted")

print("✓ All imports loaded")

---
## §3 · Synthetic Transaction Stream Generation

We simulate **500,000 transactions** over 30 days with:
- **Realistic merchant categories** (ATM, Shopping, Dining, Travel, etc.)
- **Temporal patterns** (weekday vs. weekend, intraday seasonality)
- **Geographic coordinates** (European cities)
- **Injected fraud patterns** matching the rules we'll test

In [ ]:
# ── Configuration ────────────────────────────────────────────────
N_TXN = 500_000
N_USERS = 20_000
FRAUD_RATE = 0.02  # 2% fraud rate (higher than real for demonstration)
N_FRAUD = int(N_TXN * FRAUD_RATE)
N_LEGIT = N_TXN - N_FRAUD

START = pd.Timestamp("2025-05-01")
END = pd.Timestamp("2025-05-31")
SPAN_S = (END - START).total_seconds()

CATEGORIES = ["ATM", "Shopping", "Dining", "Travel", "Grocery",
              "Entertainment", "Subscription", "Utilities", "Transfer"]

# European city coordinates
CITIES = {
    "London":    (51.51, -0.13),
    "Paris":     (48.86, 2.35),
    "Berlin":    (52.52, 13.40),
    "Madrid":    (40.42, -3.70),
    "Rome":      (41.90, 12.50),
    "Amsterdam": (52.37, 4.90),
    "Warsaw":    (52.23, 21.01),
    "New York":  (40.71, -74.01),  # for geo-impossibility
    "Lagos":     (6.52, 3.38),     # high-risk corridor
}

print(f"Generating {N_TXN:,} transactions ({FRAUD_RATE*100:.0f}% fraud rate) …")
t0 = time.time()

# ── Legitimate transactions ──────────────────────────────────────
legit_timestamps = START + pd.to_timedelta(np.random.uniform(0, SPAN_S, N_LEGIT), unit="s")
legit_user_ids = np.random.randint(1, N_USERS + 1, N_LEGIT)
legit_categories = np.random.choice(CATEGORIES, N_LEGIT,
    p=[0.15, 0.20, 0.18, 0.05, 0.15, 0.08, 0.07, 0.05, 0.07])
legit_amounts = np.round(np.abs(np.random.lognormal(3.0, 1.2, N_LEGIT)), 2)

# Assign cities (mostly European)
legit_city_names = np.random.choice(list(CITIES.keys())[:-2], N_LEGIT,
    p=[0.35, 0.15, 0.12, 0.10, 0.10, 0.10, 0.08])
legit_lats = np.array([CITIES[c][0] + np.random.normal(0, 0.05) for c in legit_city_names])
legit_lngs = np.array([CITIES[c][1] + np.random.normal(0, 0.05) for c in legit_city_names])

df_legit = pd.DataFrame({
    "txn_id":    np.arange(1, N_LEGIT + 1),
    "user_id":   legit_user_ids,
    "timestamp": legit_timestamps,
    "category":  legit_categories,
    "amount":    legit_amounts,
    "lat":       legit_lats,
    "lng":       legit_lngs,
    "city":      legit_city_names,
    "is_fraud":  0,
})

# ── Fraudulent transactions (injected patterns) ──────────────────
# Split fraud into 5 pattern types (matching our rules)
n_per_pattern = N_FRAUD // 5
fraud_dfs = []

# Pattern 1: Velocity bursts (5+ ATM withdrawals in 10 min)
burst_users = np.random.choice(range(1, N_USERS + 1), n_per_pattern // 5, replace=True)
for uid in burst_users:
    burst_start = START + pd.to_timedelta(np.random.uniform(0, SPAN_S), unit="s")
    for j in range(5):
        fraud_dfs.append({
            "txn_id": 0, "user_id": uid,
            "timestamp": burst_start + pd.Timedelta(minutes=j * 1.5),
            "category": "ATM", "amount": np.round(np.random.uniform(200, 500), 2),
            "lat": CITIES["London"][0] + np.random.normal(0, 0.01),
            "lng": CITIES["London"][1] + np.random.normal(0, 0.01),
            "city": "London", "is_fraud": 1,
        })

# Pattern 2: Geo-impossibility (London then New York in < 1 hour)
for _ in range(n_per_pattern):
    uid = np.random.randint(1, N_USERS + 1)
    ts = START + pd.to_timedelta(np.random.uniform(0, SPAN_S), unit="s")
    fraud_dfs.append({
        "txn_id": 0, "user_id": uid, "timestamp": ts,
        "category": "Shopping", "amount": np.round(np.random.uniform(500, 3000), 2),
        "lat": CITIES["London"][0], "lng": CITIES["London"][1],
        "city": "London", "is_fraud": 1,
    })
    fraud_dfs.append({
        "txn_id": 0, "user_id": uid, "timestamp": ts + pd.Timedelta(minutes=np.random.uniform(10, 50)),
        "category": "Shopping", "amount": np.round(np.random.uniform(500, 3000), 2),
        "lat": CITIES["New York"][0], "lng": CITIES["New York"][1],
        "city": "New York", "is_fraud": 1,
    })

# Pattern 3: Round-amount transfers (exactly £1000, £2000, £5000)
round_amounts = [1000.00, 2000.00, 5000.00, 10000.00]
for _ in range(n_per_pattern):
    fraud_dfs.append({
        "txn_id": 0, "user_id": np.random.randint(1, N_USERS + 1),
        "timestamp": START + pd.to_timedelta(np.random.uniform(0, SPAN_S), unit="s"),
        "category": "Transfer",
        "amount": np.random.choice(round_amounts),
        "lat": CITIES["London"][0], "lng": CITIES["London"][1],
        "city": "London", "is_fraud": 1,
    })

# Pattern 4: High-value international (Lagos, high amounts)
for _ in range(n_per_pattern):
    fraud_dfs.append({
        "txn_id": 0, "user_id": np.random.randint(1, N_USERS + 1),
        "timestamp": START + pd.to_timedelta(np.random.uniform(0, SPAN_S), unit="s"),
        "category": np.random.choice(["Shopping", "Transfer"]),
        "amount": np.round(np.random.uniform(2000, 8000), 2),
        "lat": CITIES["Lagos"][0], "lng": CITIES["Lagos"][1],
        "city": "Lagos", "is_fraud": 1,
    })

# Pattern 5: Late-night ATM (2–4 AM)
for _ in range(n_per_pattern):
    ts = START + pd.to_timedelta(np.random.uniform(0, SPAN_S), unit="s")
    ts = ts.replace(hour=np.random.randint(2, 5), minute=np.random.randint(0, 60))
    fraud_dfs.append({
        "txn_id": 0, "user_id": np.random.randint(1, N_USERS + 1),
        "timestamp": ts,
        "category": "ATM",
        "amount": np.round(np.random.uniform(300, 1000), 2),
        "lat": CITIES["London"][0] + np.random.normal(0, 0.05),
        "lng": CITIES["London"][1] + np.random.normal(0, 0.05),
        "city": "London", "is_fraud": 1,
    })

df_fraud = pd.DataFrame(fraud_dfs)

# ── Combine and shuffle ──────────────────────────────────────────
df = pd.concat([df_legit, df_fraud], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df["txn_id"] = np.arange(1, len(df) + 1)
df.sort_values("timestamp", inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"✓ Generated in {time.time() - t0:.2f}s")
print(f"  Total transactions: {len(df):,}")
print(f"  Fraud transactions : {df['is_fraud'].sum():,} ({df['is_fraud'].mean()*100:.2f}%)")
print(f"  Unique users       : {df['user_id'].nunique():,}")
print(f"  Date range         : {df['timestamp'].min()} → {df['timestamp'].max()}")

df.head(10)

---
## §4 · Exploratory Data Analysis

In [ ]:
# Summary statistics by fraud status
df.groupby("is_fraud").agg(
    count=("txn_id", "count"),
    avg_amount=("amount", "mean"),
    median_amount=("amount", "median"),
    max_amount=("amount", "max"),
    atm_pct=("category", lambda x: (x == "ATM").mean() * 100),
).round(2)

In [ ]:
# Amount distribution by fraud status
fig = px.histogram(
    df, x="amount", color="is_fraud",
    nbins=100, marginal="box",
    title="Transaction Amount Distribution (Legit vs. Fraud)",
    labels={"amount": "Amount (£)", "is_fraud": "Fraud"},
    color_discrete_map={0: "#00CC96", 1: "#EF553B"},
    barmode="overlay", opacity=0.7,
)
fig.update_xaxes(type="log", tickformat="£,.0f")
fig.update_layout(height=450)
fig.show()

In [ ]:
# Category distribution
cat_fraud = (
    df.groupby(["category", "is_fraud"])
    .size()
    .unstack(fill_value=0)
    .assign(fraud_rate=lambda d: d[1] / (d[0] + d[1]) * 100)
    .sort_values("fraud_rate", ascending=False)
    .round(2)
)
print("Fraud Rate by Category:")
print(cat_fraud)

fig = px.bar(
    cat_fraud.reset_index(),
    x="category", y="fraud_rate",
    title="Fraud Rate by Merchant Category",
    labels={"fraud_rate": "Fraud Rate (%)"},
    color="fraud_rate", color_continuous_scale="RdYlGn_r",
)
fig.update_layout(height=400, xaxis_tickangle=-30)
fig.show()

---
## §5 · Rule 1: Velocity Burst Detection

**Logic**: Flag users with **5+ transactions within a 10-minute window**.

This catches card-skimming rings that rapidly drain accounts via ATMs.

In [ ]:
def detect_velocity_bursts(df, window_minutes=10, min_txns=5):
    """
    Detect users with >= min_txns transactions within any window_minutes window.
    
    Returns a set of (user_id, txn_id) tuples that are flagged.
    """
    df_sorted = df.sort_values(["user_id", "timestamp"])
    flagged_txn_ids = set()
    
    for user_id, group in df_sorted.groupby("user_id"):
        timestamps = group["timestamp"].values
        txn_ids = group["txn_id"].values
        
        # Sliding window: for each transaction, count how many follow within the window
        for i in range(len(timestamps)):
            window_end = timestamps[i] + np.timedelta64(window_minutes, "m")
            # Count transactions in [t_i, t_i + window]
            count = np.sum((timestamps >= timestamps[i]) & (timestamps <= window_end))
            
            if count >= min_txns:
                # Flag all transactions in this window
                mask = (timestamps >= timestamps[i]) & (timestamps <= window_end)
                flagged_txn_ids.update(txn_ids[mask])
    
    return flagged_txn_ids

print("Running velocity burst detection (5+ txns in 10 min) …")
t0 = time.time()
velocity_flagged = detect_velocity_bursts(df, window_minutes=10, min_txns=5)
time_velocity = time.time() - t0

df["rule_velocity"] = df["txn_id"].isin(velocity_flagged).astype(int)

print(f"✓ Completed in {time_velocity:.2f}s")
print(f"  Flagged transactions: {df['rule_velocity'].sum():,}")
print(f"  Flagged as fraud   : {df[df['rule_velocity'] == 1]['is_fraud'].sum():,}")

# Precision and recall
prec = precision_score(df["is_fraud"], df["rule_velocity"], zero_division=0)
rec = recall_score(df["is_fraud"], df["rule_velocity"], zero_division=0)
print(f"\n  Precision: {prec:.4f}")
print(f"  Recall   : {rec:.4f}")
print(f"  F1-Score : {f1_score(df['is_fraud'], df['rule_velocity'], zero_division=0):.4f}")

---
## §6 · Rule 2: Geo-Impossibility Check

**Logic**: Flag pairs of transactions where the implied travel speed exceeds **800 km/h**
(faster than any commercial flight). This catches cloned cards used in different countries.

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    """Calculate the great-circle distance between two points (in km)."""
    R = 6371  # Earth radius in km
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    a = (np.sin(dlat / 2) ** 2 +
         np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dlon / 2) ** 2)
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c

def detect_geo_impossibility(df, max_speed_kmh=800):
    """
    Flag transactions where the user's implied travel speed exceeds max_speed_kmh.
    """
    df_sorted = df.sort_values(["user_id", "timestamp"])
    flagged_txn_ids = set()
    
    for user_id, group in df_sorted.groupby("user_id"):
        if len(group) < 2:
            continue
        
        lats = group["lat"].values
        lngs = group["lng"].values
        times = group["timestamp"].values
        txn_ids = group["txn_id"].values
        
        # Compare consecutive transactions
        for i in range(1, len(group)):
            dist_km = haversine_km(lats[i-1], lngs[i-1], lats[i], lngs[i])
            time_diff_h = (times[i] - times[i-1]) / np.timedelta64(1, "h")
            
            if time_diff_h > 0:
                speed_kmh = dist_km / time_diff_h
                if speed_kmh > max_speed_kmh:
                    flagged_txn_ids.add(txn_ids[i])
                    flagged_txn_ids.add(txn_ids[i-1])
    
    return flagged_txn_ids

print("Running geo-impossibility check (> 800 km/h) …")
t0 = time.time()
geo_flagged = detect_geo_impossibility(df, max_speed_kmh=800)
time_geo = time.time() - t0

df["rule_geo_impossible"] = df["txn_id"].isin(geo_flagged).astype(int)

print(f"✓ Completed in {time_geo:.2f}s")
print(f"  Flagged transactions: {df['rule_geo_impossible'].sum():,}")
print(f"  Flagged as fraud   : {df[df['rule_geo_impossible'] == 1]['is_fraud'].sum():,}")

prec = precision_score(df["is_fraud"], df["rule_geo_impossible"], zero_division=0)
rec = recall_score(df["is_fraud"], df["rule_geo_impossible"], zero_division=0)
print(f"\n  Precision: {prec:.4f}")
print(f"  Recall   : {rec:.4f}")
print(f"  F1-Score : {f1_score(df['is_fraud'], df['rule_geo_impossible'], zero_division=0):.4f}")

print("\n💡 Geo-impossibility has HIGH precision (few false positives)")
print("   because legitimate users rarely travel at 800+ km/h")

---
## §7 · Rule 3: Round-Amount Flag

**Logic**: Flag transfers of **exactly £1,000, £2,000, £5,000, or £10,000**.

Fraudsters often use round amounts for money laundering or structured transfers.

In [ ]:
# Define suspicious round amounts
ROUND_AMOUNTS = {1000.00, 2000.00, 5000.00, 10000.00}

# Flag transactions that are exactly round amounts (within £0.01 tolerance)
df["rule_round_amount"] = (
    df["amount"].apply(lambda x: any(abs(x - r) < 0.01 for r in ROUND_AMOUNTS))
    & (df["category"] == "Transfer")
).astype(int)

print(f"Round-Amount Rule:")
print(f"  Flagged transactions: {df['rule_round_amount'].sum():,}")
print(f"  Flagged as fraud   : {df[df['rule_round_amount'] == 1]['is_fraud'].sum():,}")

prec = precision_score(df["is_fraud"], df["rule_round_amount"], zero_division=0)
rec = recall_score(df["is_fraud"], df["rule_round_amount"], zero_division=0)
print(f"\n  Precision: {prec:.4f}")
print(f"  Recall   : {rec:.4f}")
print(f"  F1-Score : {f1_score(df['is_fraud'], df['rule_round_amount'], zero_division=0):.4f}")

# Show flagged transactions
print("\nSample flagged transactions:")
print(df[df["rule_round_amount"] == 1][["txn_id", "user_id", "amount", "category", "is_fraud"]].head(10).to_string(index=False))

---
## §8 · Rule 4: High-Value International

**Logic**: Flag transactions **> £2,000** in **high-risk corridors** (Lagos, etc.).

In [ ]:
# Define high-risk cities
HIGH_RISK_CITIES = {"Lagos"}
HIGH_VALUE_THRESHOLD = 2000.0

df["rule_high_value_intl"] = (
    (df["amount"] > HIGH_VALUE_THRESHOLD)
    & df["city"].isin(HIGH_RISK_CITIES)
).astype(int)

print(f"High-Value International Rule:")
print(f"  Flagged transactions: {df['rule_high_value_intl'].sum():,}")
print(f"  Flagged as fraud   : {df[df['rule_high_value_intl'] == 1]['is_fraud'].sum():,}")

prec = precision_score(df["is_fraud"], df["rule_high_value_intl"], zero_division=0)
rec = recall_score(df["is_fraud"], df["rule_high_value_intl"], zero_division=0)
print(f"\n  Precision: {prec:.4f}")
print(f"  Recall   : {rec:.4f}")
print(f"  F1-Score : {f1_score(df['is_fraud'], df['rule_high_value_intl'], zero_division=0):.4f}")

print("\n💡 High-risk corridor rules are highly specific but low recall")
print("   (only catches fraud in specific geographic corridors)")

---
## §9 · Rule 5: Late-Night ATM Pattern

**Logic**: Flag **ATM withdrawals between 2:00 AM and 4:00 AM** with amounts **> £300**.

Late-night large cash withdrawals correlate with mugging, drunk spending, or fraud.

In [ ]:
# Extract hour
df["hour"] = df["timestamp"].dt.hour

df["rule_late_night_atm"] = (
    (df["category"] == "ATM")
    & (df["hour"].between(2, 4))
    & (df["amount"] > 300)
).astype(int)

print(f"Late-Night ATM Rule:")
print(f"  Flagged transactions: {df['rule_late_night_atm'].sum():,}")
print(f"  Flagged as fraud   : {df[df['rule_late_night_atm'] == 1]['is_fraud'].sum():,}")

prec = precision_score(df["is_fraud"], df["rule_late_night_atm"], zero_division=0)
rec = recall_score(df["is_fraud"], df["rule_late_night_atm"], zero_division=0)
print(f"\n  Precision: {prec:.4f}")
print(f"  Recall   : {rec:.4f}")
print(f"  F1-Score : {f1_score(df['is_fraud'], df['rule_late_night_atm'], zero_division=0):.4f}")

print("\n💡 Late-night rules have moderate precision but low recall")
print("   because many legitimate users also use ATMs late at night")

---
## §10 · Combined Rule Engine

### Strategy: Flag if ANY rule fires (OR logic)

This maximizes recall at the cost of precision — suitable for a **first-pass filter**.

In [ ]:
# Combined flag: any rule fires
rule_columns = ["rule_velocity", "rule_geo_impossible", "rule_round_amount",
                "rule_high_value_intl", "rule_late_night_atm"]

df["any_rule_flagged"] = df[rule_columns].any(axis=1).astype(int)
df["num_rules_fired"] = df[rule_columns].sum(axis=1)

print("=" * 70)
print("COMBINED RULE ENGINE (OR logic)")
print("=" * 70)
print(f"\nTotal flagged      : {df['any_rule_flagged'].sum():,} ({df['any_rule_flagged'].mean()*100:.2f}%)")
print(f"True fraud flagged: {df[(df['any_rule_flagged']==1) & (df['is_fraud']==1)].shape[0]:,}")

prec_combined = precision_score(df["is_fraud"], df["any_rule_flagged"], zero_division=0)
rec_combined = recall_score(df["is_fraud"], df["any_rule_flagged"], zero_division=0)
f1_combined = f1_score(df["is_fraud"], df["any_rule_flagged"], zero_division=0)

print(f"\nCombined Precision: {prec_combined:.4f}")
print(f"Combined Recall   : {rec_combined:.4f}")
print(f"Combined F1-Score : {f1_combined:.4f}")

# Confusion matrix
cm = confusion_matrix(df["is_fraud"], df["any_rule_flagged"])
print(f"\nConfusion Matrix:")
print(f"  True Negatives : {cm[0, 0]:>8,}")
print(f"  False Positives: {cm[0, 1]:>8,}")
print(f"  False Negatives: {cm[1, 0]:>8,}")
print(f"  True Positives : {cm[1, 1]:>8,}")

In [ ]:
# ── Rule interaction: how many rules fire per fraud? ──────────────
fraud_only = df[df["is_fraud"] == 1]

print("Number of Rules Fired per Fraudulent Transaction:")
print(fraud_only["num_rules_fired"].value_counts().sort_index().to_string())

print(f"\n💡 Most fraud triggers only 1 rule — but some trigger 2–3")
print("   Multiple rules firing = higher confidence = auto-block (vs. manual review)")

---
## §11 · Precision/Recall per Rule

In [ ]:
# Compile metrics for all rules
rule_metrics = []

for col in rule_columns:
    rule_name = col.replace("rule_", "").replace("_", " ").title()
    prec = precision_score(df["is_fraud"], df[col], zero_division=0)
    rec = recall_score(df["is_fraud"], df[col], zero_division=0)
    f1 = f1_score(df["is_fraud"], df[col], zero_division=0)
    n_flagged = df[col].sum()
    n_fraud_caught = df[(df[col] == 1) & (df["is_fraud"] == 1)].shape[0]
    
    rule_metrics.append({
        "Rule": rule_name,
        "Flagged": n_flagged,
        "Fraud Caught": n_fraud_caught,
        "Precision": prec,
        "Recall": rec,
        "F1-Score": f1,
    })

# Add combined
rule_metrics.append({
    "Rule": "COMBINED (Any Rule)",
    "Flagged": df["any_rule_flagged"].sum(),
    "Fraud Caught": df[(df["any_rule_flagged"]==1) & (df["is_fraud"]==1)].shape[0],
    "Precision": prec_combined,
    "Recall": rec_combined,
    "F1-Score": f1_combined,
})

df_metrics = pd.DataFrame(rule_metrics).round(4)
print("=" * 90)
print("RULE PERFORMANCE SUMMARY")
print("=" * 90)
print(df_metrics.to_string(index=False))

---
## §12 · Visualization Dashboard

In [ ]:
# ── 12.1 Precision vs. Recall Scatter ────────────────────────────
fig = go.Figure()

for _, row in df_metrics.iterrows():
    color = "#EF553B" if row["Rule"] == "COMBINED (Any Rule)" else "#636EFA"
    size = 25 if row["Rule"] == "COMBINED (Any Rule)" else 15
    fig.add_trace(go.Scatter(
        x=[row["Recall"]], y=[row["Precision"]],
        mode="markers+text",
        text=[row["Rule"]],
        textposition="top center",
        marker=dict(size=size, color=color, line=dict(width=2, color="white")),
        showlegend=False,
    ))

fig.update_layout(
    title="Rule Performance: Precision vs. Recall",
    xaxis_title="Recall (Fraud Detection Rate)",
    yaxis_title="Precision (True Fraud % of Flags)",
    height=500,
    xaxis_range=[-0.05, 1.05],
    yaxis_range=[-0.05, 1.05],
)
fig.show()

print("💡 Top-right = best (high precision AND high recall)")
print("   Bottom-right = catches lots of fraud but many false alarms")
print("   Top-left = very precise but misses most fraud")

In [ ]:
# ── 12.2 Rule Contribution Stacked Bar ───────────────────────────
fig = go.Figure()

for col in rule_columns:
    rule_name = col.replace("rule_", "").replace("_", " ").title()
    n_fraud = df[(df[col] == 1) & (df["is_fraud"] == 1)].shape[0]
    n_legit = df[(df[col] == 1) & (df["is_fraud"] == 0)].shape[0]
    
    fig.add_trace(go.Bar(
        name=rule_name,
        x=["Flagged Transactions"],
        y=[n_fraud],
        marker_color=px.colors.qualitative.Set2[rule_columns.index(col)],
    ))

fig.update_layout(
    title="Fraud Caught by Each Rule",
    yaxis_title="Number of Fraudulent Transactions Caught",
    barmode="stack",
    height=400,
)
fig.show()

In [ ]:
# ── 12.3 Business Impact Analysis ────────────────────────────────
# Assumptions
AVG_FRAUD_AMOUNT = df[df["is_fraud"]==1]["amount"].mean()
REVIEW_COST = 5.0    # £ per manual review
FRAUD_LOSS = AVG_FRAUD_AMOUNT  # £ lost per missed fraud

total_fraud = df["is_fraud"].sum()
true_positives = df[(df["any_rule_flagged"]==1) & (df["is_fraud"]==1)].shape[0]
false_positives = df[(df["any_rule_flagged"]==1) & (df["is_fraud"]==0)].shape[0]
false_negatives = df[(df["any_rule_flagged"]==0) & (df["is_fraud"]==1)].shape[0]

fraud_prevented_value = true_positives * AVG_FRAUD_AMOUNT
review_cost_total = false_positives * REVIEW_COST
fraud_missed_cost = false_negatives * AVG_FRAUD_AMOUNT
net_value = fraud_prevented_value - review_cost_total

print("=" * 70)
print("BUSINESS IMPACT ANALYSIS (Monthly, 500K transactions)")
print("=" * 70)
print(f"\nAssumptions:")
print(f"  Average fraud amount : £{AVG_FRAUD_AMOUNT:,.2f}")
print(f"  Manual review cost   : £{REVIEW_COST:.2f} per transaction")
print(f"\nRule Engine Performance:")
print(f"  True Positives (fraud caught)   : {true_positives:,}")
print(f"  False Positives (legit flagged)   : {false_positives:,}")
print(f"  False Negatives (fraud missed)    : {false_negatives:,}")
print(f"\nFinancial Impact:")
print(f"  Fraud prevented (value saved)   : £{fraud_prevented_value:,.0f}")
print(f"  Manual review cost              : £{review_cost_total:,.0f}")
print(f"  Fraud missed (opportunity cost) : £{fraud_missed_cost:,.0f}")
print(f"  NET VALUE                       : £{net_value:,.0f}")

print(f"\n💡 The rule engine saves £{net_value:,.0f}/month on 500K transactions")
print(f"   by catching {true_positives:,} fraud cases with only {false_positives:,} false alarms")

In [ ]:
# ── 12.4 Rule Coverage Heatmap ───────────────────────────────────
# Show which fraud patterns are caught by which rules
fraud_df = df[df["is_fraud"] == 1].copy()

# Assign fraud pattern type based on characteristics
fraud_df["pattern"] = "Unknown"
fraud_df.loc[fraud_df["rule_velocity"] == 1, "pattern"] = "Velocity Burst"
fraud_df.loc[fraud_df["rule_geo_impossible"] == 1, "pattern"] = "Geo-Impossible"
fraud_df.loc[fraud_df["rule_round_amount"] == 1, "pattern"] = "Round Amount"
fraud_df.loc[fraud_df["rule_high_value_intl"] == 1, "pattern"] = "High-Value Intl"
fraud_df.loc[fraud_df["rule_late_night_atm"] == 1, "pattern"] = "Late-Night ATM"

pattern_counts = fraud_df["pattern"].value_counts()
fig = px.pie(
    values=pattern_counts.values,
    names=pattern_counts.index,
    title="Fraud Patterns Caught by Rules",
    hole=0.4,
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.update_layout(height=400)
fig.show()

---
## §13 · Key Business Takeaway

> ### 🎯 Action Items for a Fintech Data Scientist
>
> | Rule | Precision | Recall | Best For |
> |------|-----------|--------|----------|
> | **Velocity Burst** | High | Medium | Card skimming, account takeover |
> | **Geo-Impossibility** | Very High | Low | Cloned cards, credential theft |
> | **Round Amount** | Medium | Low | Money laundering, structured fraud |
> | **High-Value Intl** | High | Low | High-risk corridors, mule networks |
> | **Late-Night ATM** | Medium | Low | Mugging, drunk spending, fraud |
> | **COMBINED (OR)** | Medium | High | First-pass filter before ML |
>
> ### 💡 Revolut-Specific Guidelines
>
> 1. **Rules are the first line of defense** — they run in < 5 ms and catch known patterns.
>    - Use rules for **auto-blocking** high-confidence fraud (e.g., geo-impossibility).
>    - Use rules for **manual review queue** for medium-confidence flags.
>
> 2. **Rule thresholds must be tuned quarterly**:
>    - Velocity: adjust window size and min transactions based on fraud trends.
>    - Geo: adjust speed threshold based on new flight routes.
>    - Amounts: adjust round amounts based on inflation and new limits.
>
> 3. **Combine rules with ML**:
>    ```
>    Transaction → Rule Engine (5 ms) → Flagged? → Auto-block / Review
>                                      ↓ Not flagged
>                               ML Model (50 ms) → Score → Decision
>    ```
>
> 4. **Monitor rule decay**: rules lose effectiveness as fraudsters adapt.
>    - Track precision/recall weekly.
>    - Retire rules with < 10% precision (too many false alarms).
>
> ### 📌 Rule Engine Design Principles
>
> 1. **Fast**: < 5 ms per transaction (no database lookups, pure in-memory).
> 2. **Interpretable**: every flag has a clear reason for compliance.
> 3. **Tunable**: thresholds adjustable without code changes.
> 4. **Layered**: multiple rules with OR logic for recall, AND logic for precision.
> 5. **Monitored**: dashboards showing precision, recall, and volume per rule.
>
> ### 🔑 Key Takeaways
>
> 1. **Rules catch 30–40% of fraud** with minimal latency — essential for real-time systems.
> 2. **Geo-impossibility has the highest precision** — use for auto-blocking.
> 3. **Velocity bursts have the highest recall** — use for review queues.
> 4. **Combined OR logic maximizes recall** — suitable for a first-pass filter.
> 5. **Rules complement ML** — they catch known patterns that ML may miss in low-data regimes.